### Export diffusion_results_* → per-state CSVs (parallel) and combine by state

In [ ]:
# Set path to access helpers
from pathlib import Path
import sys

PARENT = Path.cwd().parent     # helpers live here
if str(PARENT) not in sys.path:
    sys.path.insert(0, str(PARENT))

In [ ]:
from schema_exporter import ConnParams, export_all

In [ ]:
OUT_DIR = "/Users/wael/Library/CloudStorage/GoogleDrive-wael@permitpower.org/Shared drives/PP (All)/Research/$1 watt solar/2025/Results/Updated tariffs"

# ---- switch the whole export by changing ATTACH_PCT only -------------------
ATTACH_PCT = 75            # 5 | 75 | 100  (the model tags schemas _a<pct>)
SCHEMA_TAG = f"_a{ATTACH_PCT}_"
RUN_ID     = f"synapse_attachrate_{ATTACH_PCT}"
# ---------------------------------------------------------------------------

# Synapse deliverable: flat battery attachment (FLAT_STORAGE_ATTACHMENT_RATE).
# All three runs also share:
#   - state-specific LBNL-2025 baseline PV costs (13 states own median, 36 national $3.62/W)
#   - federal ITC removed entirely, and no 0.7x battery discount
#   - Washington DC included -> 49 states
# Adoption is IDENTICAL across the three rates (attachment only sorts adopters into
# the PV-only vs PV+battery buckets afterwards), so row counts should match exactly.
#
# Prior, unrelated runs: nj_srec | run_all_states_updated_tariffs (immediate $1/W)
#
# Results land in {OUT_DIR}/{STATE}/{RUN_ID}/


In [ ]:
# Discover this rate's schemas rather than pasting 98 names.
# NOTE: never leave schemas_include=None -- export_all would then export EVERY
# diffusion_results_* schema in the DB, including the older nj_srec / $1-per-watt studies.
import psycopg2

cp = ConnParams.from_env()

con = psycopg2.connect(dbname=cp.dbname, user=cp.user, password=cp.password,
                       host=cp.host, port=cp.port)
cur = con.cursor()
cur.execute(
    "select nspname from pg_namespace where nspname like %s order by nspname",
    (f"diffusion_results_%{SCHEMA_TAG}%",),
)
SCHEMAS = [r[0] for r in cur.fetchall()]
con.close()

n_base = sum(1 for s in SCHEMAS if s.split("_")[2] == "baseline")
n_pol  = sum(1 for s in SCHEMAS if s.split("_")[2] == "policy")
n_st   = len({s.split("_")[3] for s in SCHEMAS})
print(f"{RUN_ID}: {len(SCHEMAS)} schemas -- {n_base} baseline, {n_pol} policy, {n_st} states")

# Guard: expect exactly one complete schema per state per scenario.
assert (len(SCHEMAS), n_base, n_pol, n_st) == (98, 49, 49, 49), \
    f"unexpected schema set: {len(SCHEMAS)} total / {n_base} base / {n_pol} pol / {n_st} states"

summary = export_all(
    cp=cp,
    out_dir=OUT_DIR,
    run_id=RUN_ID,
    chunksize=200_000,
    jobs=10,
    overwrite=True,
    only_scenarios=None,
    states_filter=None,
    schemas_include=SCHEMAS,
)

print("exported:", len(summary["exported"]),
      "| skipped:", len(summary["skipped"]),
      "| failed:", summary["failed"],
      "| missing:", summary["missing"])
summary
